# Demo 2 — End-to-end prediction

Predict a full duty cycle from an origin/destination pair with `DutyCyclePredictor.predict()`:
HERE route → per-second speed profile → SRF elevation/gradient profile → energy/fuel profile.

**Prerequisites**:

- dependencies from `requirements.txt` (conda env `dcp`) — no installation needed;
- `HERE_API_KEY` in the repository-root `.env` (**required** — copy `.env.example` to `.env`);
- `SRF_API_KEY` in `.env` (*optional* — without it the gradient profile falls back to flat).

`predict()` returns `None` when the route is shorter than ~5 km (a guard against degenerate routes).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))  # repo root - makes `import dcpredictor` resolve without installation

from datetime import datetime

import folium
import matplotlib.pyplot as plt

from dcpredictor import (
    DutyCyclePredictor,
    load_default_driving_behavior,
    load_default_vehicle_params,
)

## 1. Run the prediction

`load_default_vehicle_params()` / `load_default_driving_behavior()` load presets from the package's
`params/*.json` (`"default"`, or a vehicle registration such as `"AY71UCD"` / `"FX73VAE"`). Override any
field by constructing `VehicleParams` / `DrivingBehavior` directly.

In [ ]:
predictor = DutyCyclePredictor()  # reads API keys from .env

result = predictor.predict(
    origin=(52.292, 0.389),        # (lat, lon)
    destination=(51.550, -0.242),
    mass_kg=5000.0,
    vehicle_params=load_default_vehicle_params(),
    driving_behavior=load_default_driving_behavior(),
    departure_time=datetime(2025, 2, 27, 9, 0, 0),
)

assert result is not None, "Route too short or invalid - no duty cycle generated."
print(f"Speed profile   : {len(result.speed_profile)} steps")
print(f"Gradient profile: {len(result.gradient_profile)} points")
print(f"Energy profile  : {len(result.energy_profile)} points")

## 2. Inspect the three profiles

The result is a `DutyCycle` dataclass holding three row-aligned DataFrames.

In [ ]:
result.speed_profile.head()

In [ ]:
result.gradient_profile.head()

In [ ]:
result.energy_profile.head()

## 3. Plot the duty cycle

In [ ]:
distance_km = result.speed_profile["distance"] / 1000

fig, (ax_v, ax_g, ax_f) = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

ax_v.plot(distance_km, result.speed_profile["speed"] * 3.6, color="tab:blue")
ax_v.set_ylabel("Speed (km/h)")
ax_v.set_title("Predicted duty cycle")
ax_v.grid(True)

ax_g.plot(distance_km, result.gradient_profile["elevation"], color="tab:brown", label="Elevation (m)")
ax_g.set_ylabel("Elevation (m)")
ax_g.grid(True)
ax_g2 = ax_g.twinx()
ax_g2.plot(distance_km, result.gradient_profile["gradient"], color="tab:orange", alpha=0.6, label="Gradient (deg)")
ax_g2.set_ylabel("Gradient (deg)")

ax_f.plot(distance_km, result.energy_profile["fuel_rate_L_hr"], color="tab:green", label="Fuel rate (L/hr)")
ax_f.set_xlabel("Distance (km)")
ax_f.set_ylabel("Fuel rate (L/hr)")
ax_f.grid(True)
ax_f2 = ax_f.twinx()
ax_f2.plot(distance_km, result.energy_profile["fuel_cumulative_L"], color="tab:red", label="Cumulative fuel (L)")
ax_f2.set_ylabel("Cumulative fuel (L)")

plt.tight_layout()
plt.show()

total_fuel_L = result.energy_profile["fuel_cumulative_L"].iloc[-1]
print(f"Distance: {distance_km.iloc[-1]:.1f} km | Total fuel: {total_fuel_L:.2f} L")

## 4. Route trajectory on the map

The predicted route rendered with folium (OpenStreetMap tiles, no extra API key). The interactive HTML
is also saved to `results/` (gitignored).

In [ ]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

route_coords = result.speed_profile[["Lat", "Lon"]].dropna().to_numpy().tolist()

m = folium.Map(location=route_coords[len(route_coords) // 2], zoom_start=9)
folium.PolyLine(route_coords, color="blue", weight=4, opacity=0.8, popup="Predicted route").add_to(m)
folium.Marker(route_coords[0], popup="Origin", icon=folium.Icon(color="green", icon="play")).add_to(m)
folium.Marker(route_coords[-1], popup="Destination", icon=folium.Icon(color="red", icon="stop")).add_to(m)
m.add_child(folium.LatLngPopup())

map_path = RESULTS_DIR / "predicted_route_map.html"
m.save(str(map_path))
print(f"Map saved: {map_path}")
m

## Next steps

- **Demo 3** (`predict_vs_measured_leg.ipynb`) — validate a prediction against a measured GPS trip leg.
- Package internals (API, module map, algorithms): [`../dcpredictor/README.md`](../dcpredictor/README.md).